# Multimodal Confusion Pattern Analysis

Minimal notebook to analyze confusion matrices across model variants.

- Assumes this notebook lives in `analysis/multimodal/`
- Assumes prediction files live in `outputs_multimodal/*/Dy*/organoid_predictions.csv`
- Focus: model/backbone/input/fusion, **not** day.

## 1. Setup

In [6]:
from pathlib import Path
import numpy as np
import pandas as pd


## 2. Load predictions and parse model metadata

In [7]:
# Root relative to this notebook
ROOT = Path("outputs_multimodal")

model_dirs = [
    d for d in ROOT.iterdir()
    if d.is_dir() and d.name not in {"overall", "logs"}
]

print("Found model directories:")
for d in model_dirs:
    print(" -", d.name)

all_preds = []

for model_dir in model_dirs:
    model_id = model_dir.name  # e.g. 'efficientnet_rgb_mask_concat'
    for day_dir in model_dir.iterdir():
        if not day_dir.is_dir():
            continue
        csv_path = day_dir / "organoid_predictions.csv"
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)
        df["Model_ID"] = model_id
        all_preds.append(df)

if not all_preds:
    raise RuntimeError("No organoid_predictions.csv files found under outputs_multimodal.")

preds = pd.concat(all_preds, ignore_index=True)
print("Total prediction rows:", len(preds))


Found model directories:
 - efficientnet_rgb_concat
 - resnet_rgb_mask_gated
 - resnet_rgb_gated
 - efficientnet_rgb_gated
 - efficientnet_rgb_mask_concat
 - resnet_rgb_mask_concat
 - resnet_rgb_concat
 - efficientnet_rgb_mask_gated
Total prediction rows: 3320


In [8]:
def parse_model_id(mid: str):
    parts = mid.split("_")
    backbone = parts[0]
    if len(parts) == 3:
        input_mode = parts[1]
        fusion = parts[2]
    elif len(parts) == 4:
        input_mode = parts[1] + "_" + parts[2]
        fusion = parts[3]
    else:
        input_mode = "unknown"
        fusion = "unknown"
    return backbone, input_mode, fusion

parsed = preds["Model_ID"].apply(parse_model_id)
preds["Backbone"] = parsed.apply(lambda x: x[0])
preds["Input_Mode"] = parsed.apply(lambda x: x[1])
preds["Fusion"] = parsed.apply(lambda x: x[2])

# Treat each Organoid_ID as one cell across models
preds["Cell_ID"] = preds["Organoid_ID"].astype(str)

print("Backbones:", preds["Backbone"].unique())
print("Input modes:", preds["Input_Mode"].unique())
print("Fusion strategies:", preds["Fusion"].unique())
print("Unique cells:", preds["Cell_ID"].nunique())


Backbones: ['efficientnet' 'resnet']
Input modes: ['rgb' 'rgb_mask']
Fusion strategies: ['concat' 'gated']
Unique cells: 44


## 3. Confusion statistics by model and by backbone/input/fusion

In [12]:
import numpy as np

# Per-model confusion totals via value_counts
cm = (
    preds.groupby("Model_ID")["CM_Category"]
    .value_counts()
    .unstack(fill_value=0)        # columns like 'TP','FP','TN','FN' (whatever is present)
)

# Ensure all four columns exist in a fixed order
cm = cm.reindex(columns=["TP", "FP", "TN", "FN"], fill_value=0)

model_cm = cm.reset_index()

# Metrics
model_cm["accuracy"] = (model_cm["TP"] + model_cm["TN"]) / (
    model_cm["TP"] + model_cm["TN"] + model_cm["FP"] + model_cm["FN"]
)
model_cm["precision"] = model_cm["TP"] / (model_cm["TP"] + model_cm["FP"]).replace(0, np.nan)
model_cm["recall"] = model_cm["TP"] / (model_cm["TP"] + model_cm["FN"]).replace(0, np.nan)
model_cm["specificity"] = model_cm["TN"] / (model_cm["TN"] + model_cm["FP"]).replace(0, np.nan)

print("\nPer-model confusion summary (sorted by accuracy):")
display(model_cm.sort_values("accuracy", ascending=False))



Per-model confusion summary (sorted by accuracy):


CM_Category,Model_ID,TP,FP,TN,FN,accuracy,precision,recall,specificity
2,efficientnet_rgb_mask_concat,255,29,45,86,0.722892,0.897887,0.747801,0.608108
0,efficientnet_rgb_concat,241,27,47,100,0.693976,0.899254,0.706745,0.635135
6,resnet_rgb_mask_concat,241,32,42,100,0.681928,0.882784,0.706745,0.567568
3,efficientnet_rgb_mask_gated,243,39,35,98,0.669880,0.861702,0.712610,0.472973
1,efficientnet_rgb_gated,236,33,41,105,0.667470,0.877323,0.692082,0.554054
4,resnet_rgb_concat,229,28,46,112,0.662651,0.891051,0.671554,0.621622
7,resnet_rgb_mask_gated,230,31,43,111,0.657831,0.881226,0.674487,0.581081
5,resnet_rgb_gated,224,36,38,117,0.631325,0.861538,0.656891,0.513514


In [14]:

# Per (backbone, input_mode, fusion) confusion totals via value_counts
fam = (
    preds.groupby(["Backbone", "Input_Mode", "Fusion"])["CM_Category"]
    .value_counts()
    .unstack(fill_value=0)   # columns like 'TP','FP','TN','FN' if present
)

# Ensure all four confusion columns exist
fam = fam.reindex(columns=["TP", "FP", "TN", "FN"], fill_value=0)

family_cm = fam.reset_index()

# Metrics
family_cm["accuracy"] = (family_cm["TP"] + family_cm["TN"]) / (
    family_cm["TP"] + family_cm["TN"] + family_cm["FP"] + family_cm["FN"]
)
family_cm["precision"] = family_cm["TP"] / (family_cm["TP"] + family_cm["FP"]).replace(0, np.nan)
family_cm["recall"] = family_cm["TP"] / (family_cm["TP"] + family_cm["FN"]).replace(0, np.nan)
family_cm["specificity"] = family_cm["TN"] / (family_cm["TN"] + family_cm["FP"]).replace(0, np.nan)

print("\nPer-backbone/input/fusion confusion summary (sorted by accuracy):")
display(family_cm.sort_values("accuracy", ascending=False))



Per-backbone/input/fusion confusion summary (sorted by accuracy):


CM_Category,Backbone,Input_Mode,Fusion,TP,FP,TN,FN,accuracy,precision,recall,specificity
2,efficientnet,rgb_mask,concat,255,29,45,86,0.722892,0.897887,0.747801,0.608108
0,efficientnet,rgb,concat,241,27,47,100,0.693976,0.899254,0.706745,0.635135
6,resnet,rgb_mask,concat,241,32,42,100,0.681928,0.882784,0.706745,0.567568
3,efficientnet,rgb_mask,gated,243,39,35,98,0.669880,0.861702,0.712610,0.472973
1,efficientnet,rgb,gated,236,33,41,105,0.667470,0.877323,0.692082,0.554054
4,resnet,rgb,concat,229,28,46,112,0.662651,0.891051,0.671554,0.621622
7,resnet,rgb_mask,gated,230,31,43,111,0.657831,0.881226,0.674487,0.581081
5,resnet,rgb,gated,224,36,38,117,0.631325,0.861538,0.656891,0.513514


## 4. Cell-level consistency across all models

In [15]:
cell_cm = (
    preds.groupby("Cell_ID")
    .agg(
        n_runs=("Model_ID", "count"),
        TP=("CM_Category", lambda s: (s == "TP").sum()),
        FP=("CM_Category", lambda s: (s == "FP").sum()),
        TN=("CM_Category", lambda s: (s == "TN").sum()),
        FN=("CM_Category", lambda s: (s == "FN").sum()),
    )
)

cell_cm["frac_TP"] = cell_cm["TP"] / cell_cm["n_runs"]
cell_cm["frac_FP"] = cell_cm["FP"] / cell_cm["n_runs"]
cell_cm["frac_TN"] = cell_cm["TN"] / cell_cm["n_runs"]
cell_cm["frac_FN"] = cell_cm["FN"] / cell_cm["n_runs"]

print("Total cells:", len(cell_cm))
cell_cm.head()


Total cells: 44


,n_runs,TP,FP,TN,FN,frac_TP,frac_FP,frac_TN,frac_FN
Cell_ID,,,,,,,,,
BA1 96_1 A10,80,68,0,0,12,0.8500,0.000,0.000,0.1500
BA1 96_1 B3,80,70,0,0,10,0.8750,0.000,0.000,0.1250
BA1 96_1 B9,80,72,0,0,8,0.9000,0.000,0.000,0.1000
BA1 96_1 C10,80,65,0,0,15,0.8125,0.000,0.000,0.1875
BA1 96_1 C7,80,0,66,14,0,0.0000,0.825,0.175,0.0000


In [16]:
EASY_THRESH = 0.90
HARD_THRESH = 0.50

easy_cells = cell_cm[cell_cm["frac_TP"] >= EASY_THRESH]
hard_FN_cells = cell_cm[cell_cm["frac_FN"] >= HARD_THRESH]
hard_FP_cells = cell_cm[cell_cm["frac_FP"] >= HARD_THRESH]

print("Easy cells (mostly TP):", len(easy_cells))
print("Hard FN cells (often missed):", len(hard_FN_cells))
print("Hard FP cells (often false alarms):", len(hard_FP_cells))


Easy cells (mostly TP): 7
Hard FN cells (often missed): 6
Hard FP cells (often false alarms): 4


## 5. Inspect example hard and easy cells

In [17]:
print("Top hard FN cells:")
display(hard_FN_cells.sort_values("frac_FN", ascending=False).head(10))

print("Top hard FP cells:")
display(hard_FP_cells.sort_values("frac_FP", ascending=False).head(10))

print("Sample easy cells:")
display(easy_cells.sort_values("frac_TP", ascending=False).head(10))


Top hard FN cells:


,n_runs,TP,FP,TN,FN,frac_TP,frac_FP,frac_TN,frac_FN
Cell_ID,,,,,,,,,
BA2 96_1 F10,88,30,0,0,58,0.340909,0.0,0.0,0.659091
BA2 96_2 B6,88,35,0,0,53,0.397727,0.0,0.0,0.602273
BA2 96_2 E6,88,36,0,0,52,0.409091,0.0,0.0,0.590909
BA2 96_2 E10,88,37,0,0,51,0.420455,0.0,0.0,0.579545
BA2 96_1 E7 split_1,32,14,0,0,18,0.437500,0.0,0.0,0.562500
BA2 96_1 C7,88,44,0,0,44,0.500000,0.0,0.0,0.500000


Top hard FP cells:


,n_runs,TP,FP,TN,FN,frac_TP,frac_FP,frac_TN,frac_FN
Cell_ID,,,,,,,,,
BA2 96_2 B3 split_2,24,0,22,2,0,0.0,0.916667,0.083333,0.0
BA1 96_1 C7,80,0,66,14,0,0.0,0.825000,0.175000,0.0
BA2 96_1 A12,88,0,51,37,0,0.0,0.579545,0.420455,0.0
BA2 96_2 B8 split_2,24,0,13,11,0,0.0,0.541667,0.458333,0.0


Sample easy cells:


,n_runs,TP,FP,TN,FN,frac_TP,frac_FP,frac_TN,frac_FN
Cell_ID,,,,,,,,,
BA1 96_1 D8,80,77,0,0,3,0.962500,0.0,0.0,0.037500
BA1 96_1 F5,80,77,0,0,3,0.962500,0.0,0.0,0.037500
BA2 96_2 B5 split_2,24,23,0,0,1,0.958333,0.0,0.0,0.041667
BA1 96_1 G9,80,76,0,0,4,0.950000,0.0,0.0,0.050000
BA1 96_1 F3,80,73,0,0,7,0.912500,0.0,0.0,0.087500
BA1 96_1 F8,80,73,0,0,7,0.912500,0.0,0.0,0.087500
BA1 96_1 B9,80,72,0,0,8,0.900000,0.0,0.0,0.100000


In [18]:
# Filter only this model
eff_mask_concat = preds[preds["Model_ID"] == "efficientnet_rgb_mask_concat"]

# Compute cell-level confusion for this model only
eff_cell = (
    eff_mask_concat.groupby("Cell_ID")
    .agg(
        n_runs=("Model_ID", "count"),
        TP=("CM_Category", lambda s: (s=="TP").sum()),
        FP=("CM_Category", lambda s: (s=="FP").sum()),
        TN=("CM_Category", lambda s: (s=="TN").sum()),
        FN=("CM_Category", lambda s: (s=="FN").sum()),
    )
)

eff_cell["frac_TP"] = eff_cell["TP"] / eff_cell["n_runs"]
eff_cell["frac_FP"] = eff_cell["FP"] / eff_cell["n_runs"]
eff_cell["frac_TN"] = eff_cell["TN"] / eff_cell["n_runs"]
eff_cell["frac_FN"] = eff_cell["FN"] / eff_cell["n_runs"]

# Hard and easy subsets
hard_FN_eff = eff_cell[eff_cell["frac_FN"] > 0]       # FN > 0 means it ever missed
hard_FP_eff = eff_cell[eff_cell["frac_FP"] > 0]       # FP > 0 means it ever falsely predicted positive
easy_eff = eff_cell[eff_cell["frac_TP"] == 1]         # Perfect TP cells for this model

print("Top hard FN cells for efficientnet_rgb_mask_concat:")
display(hard_FN_eff.sort_values("frac_FN", ascending=False).head(10))

print("Top hard FP cells for efficientnet_rgb_mask_concat:")
display(hard_FP_eff.sort_values("frac_FP", ascending=False).head(10))

print("Sample easy cells for efficientnet_rgb_mask_concat:")
display(easy_eff.sort_values("frac_TP", ascending=False).head(10))


Top hard FN cells for efficientnet_rgb_mask_concat:


,n_runs,TP,FP,TN,FN,frac_TP,frac_FP,frac_TN,frac_FN
Cell_ID,,,,,,,,,
BA2 96_1 F10,11,3,0,0,8,0.272727,0.0,0.0,0.727273
BA2 96_1 E1 split_2,3,1,0,0,2,0.333333,0.0,0.0,0.666667
BA2 96_2 B6,11,5,0,0,6,0.454545,0.0,0.0,0.545455
BA2 96_1 E7 split_1,4,2,0,0,2,0.500000,0.0,0.0,0.500000
BA2 96_2 E10,11,6,0,0,5,0.545455,0.0,0.0,0.454545
BA2 96_1 C7,11,7,0,0,4,0.636364,0.0,0.0,0.363636
BA2 96_1 C10,11,7,0,0,4,0.636364,0.0,0.0,0.363636
BA2 96_1 E2,11,7,0,0,4,0.636364,0.0,0.0,0.363636
BA2 96_2 A9,11,7,0,0,4,0.636364,0.0,0.0,0.363636


Top hard FP cells for efficientnet_rgb_mask_concat:


,n_runs,TP,FP,TN,FN,frac_TP,frac_FP,frac_TN,frac_FN
Cell_ID,,,,,,,,,
BA2 96_2 B3 split_2,3,0,3,0,0,0.0,1.000000,0.000000,0.0
BA1 96_1 C7,10,0,9,1,0,0.0,0.900000,0.100000,0.0
BA2 96_1 A12,11,0,7,4,0,0.0,0.636364,0.363636,0.0
BA2 96_1 B3,11,0,3,8,0,0.0,0.272727,0.727273,0.0
BA2 96_1 B8,11,0,3,8,0,0.0,0.272727,0.727273,0.0
BA2 96_1 E9,11,0,3,8,0,0.0,0.272727,0.727273,0.0
BA2 96_2 G4,11,0,1,10,0,0.0,0.090909,0.909091,0.0


Sample easy cells for efficientnet_rgb_mask_concat:


,n_runs,TP,FP,TN,FN,frac_TP,frac_FP,frac_TN,frac_FN
Cell_ID,,,,,,,,,
BA1 96_1 D8,10,10,0,0,0,1.0,0.0,0.0,0.0
BA1 96_1 G12,10,10,0,0,0,1.0,0.0,0.0,0.0
BA1 96_1 G9,10,10,0,0,0,1.0,0.0,0.0,0.0
BA2 96_2 B5 split_2,3,3,0,0,0,1.0,0.0,0.0,0.0
